# Learning 03: Agents and Tools

Agents use an LLM to dynamically decide which tools to call. Unlike chains (fixed sequence), agents loop until the task is complete.

**LangChain 1.x API**: Use `create_agent` from `langchain.agents`. The agent accepts messages and returns messages.

## Setup

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

OPENAI_API_KEY = "your-api-key-here"
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=OPENAI_API_KEY)
print("✓ LLM initialized")


In [ ]:
import json, os

# In Jupyter, os.getcwd() returns the notebook's directory (learning/, tasks/, solutions/)
# Fixtures are one level up at ../fixtures/input/
fixtures = os.path.normpath(os.path.join(os.getcwd(), "..", "fixtures", "input"))

with open(os.path.join(fixtures, "tickets.json")) as f:
    tickets = json.load(f)

with open(os.path.join(fixtures, "knowledge_base.json")) as f:
    knowledge_base = json.load(f)

with open(os.path.join(fixtures, "test_questions.json")) as f:
    test_questions = json.load(f)

print(f"✓ Loaded {len(tickets)} tickets, {len(knowledge_base)} KB articles, {len(test_questions)} test questions")


## 1. Define Tools with `@tool`

The **docstring** is what the agent reads to decide when to use the tool. Make it specific.

In [ ]:
import json
from langchain_core.tools import tool

@tool
def classify_ticket(ticket_text: str) -> str:
    """Classify a customer support ticket by category and priority.
    Use this when you need to determine what type of issue a ticket describes
    and how urgent it is.

    Args:
        ticket_text: The full text of the support ticket.

    Returns:
        JSON string with 'category' (billing/technical/account/shipping)
        and 'priority' (high/medium/low) fields.
    """
    prompt = ChatPromptTemplate.from_messages([
        ("system", 'Return ONLY valid JSON: {{"category": "billing|technical|account|shipping", "priority": "high|medium|low"}}'),
        ("human", "{ticket}"),
    ])
    chain = prompt | llm | JsonOutputParser()
    result = chain.invoke({"ticket": ticket_text})
    return json.dumps(result)

# Test the tool directly
print(classify_ticket.invoke({"ticket_text": "I was charged twice this month"}))


In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

texts = [article["content"] for article in knowledge_base]
metadatas = [{"id": a["id"], "title": a["title"]} for a in knowledge_base]
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small", api_key=OPENAI_API_KEY)
vectorstore = FAISS.from_texts(texts, embeddings_model, metadatas=metadatas)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

@tool
def search_knowledge_base(query: str) -> str:
    """Search the company knowledge base for policy and troubleshooting information.
    Use this when the customer asks about refunds, shipping, API rate limits, account
    issues, password reset, cancellation, team management, or any company procedures.

    Args:
        query: The customer's question or search terms.

    Returns:
        Relevant knowledge base articles as formatted text.
    """
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant articles found."
    return "\n\n".join(
        f"[Article {i+1}: {doc.metadata['title']}]\n{doc.page_content}"
        for i, doc in enumerate(docs)
    )

print(search_knowledge_base.invoke({"query": "API rate limit 429 error"}))


In [ ]:
@tool
def escalate_to_human(reason: str, priority: str) -> str:
    """Escalate a support ticket to a human agent.
    Use this when: (1) the issue is urgent (priority=high) and involves account
    compromise, data loss, or financial damage; (2) you cannot find the answer
    in the knowledge base; (3) the customer is frustrated and needs human empathy.

    Args:
        reason: Brief explanation of why escalation is needed.
        priority: The urgency level (high/medium/low).

    Returns:
        Confirmation that the ticket has been escalated.
    """
    return f"✓ Ticket escalated to human agent (priority={priority}). Reason: {reason}"

tools = [classify_ticket, search_knowledge_base, escalate_to_human]
print(f"✓ Defined {len(tools)} tools: {[t.name for t in tools]}")


## 2. Build the Agent

In **LangChain 1.x**, use `create_agent` from `langchain.agents`.
It takes `(model, tools, system_prompt=...)` and returns a LangGraph agent.

The agent accepts `{"messages": [...]}` and returns `{"messages": [...]}`.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    llm,
    tools,
    system_prompt="""\
You are a customer support agent for a SaaS company.
Your job is to help customers by:
1. Classifying their issue
2. Searching the knowledge base for relevant information
3. Providing a clear, helpful answer
4. Escalating to a human if the issue is urgent or you cannot help

Always classify the ticket first, then search for information.""",
)
print("✓ Agent ready")


## 3. Run the Agent

Invoke with a messages list. The final answer is in the last message.

In [ ]:
# Easy question — should search KB and answer directly
result = agent.invoke({
    "messages": [{"role": "user", "content": "Why is my API returning 429 errors?"}]
})

final_answer = result["messages"][-1].content
print("Final answer:")
print(final_answer)


In [ ]:
# Urgent — should classify as high priority and escalate
result = agent.invoke({
    "messages": [{"role": "user", "content": "My account was hacked! I see login attempts from Russia. Lock it now!"}]
})
print("Final answer:", result["messages"][-1].content)


In [ ]:
# Policy question — should search KB
result = agent.invoke({
    "messages": [{"role": "user", "content": "What happens to my data if I cancel my subscription?"}]
})
print("Final answer:", result["messages"][-1].content)


## 4. Inspect Tool Calls

All messages (user, tool calls, tool results, assistant) are in `result["messages"]`.

In [ ]:
from langchain_core.messages import ToolMessage, AIMessage

result = agent.invoke({
    "messages": [{"role": "user", "content": "I ordered a laptop stand but received a mouse pad. Order #45123."}]
})

print("All messages:")
for msg in result["messages"]:
    name = type(msg).__name__
    content = str(msg.content)[:80] if msg.content else ""
    # AIMessage may have tool_calls
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  [{name}] → calls tool: {tc['name']}({tc['args']})")
    elif isinstance(msg, ToolMessage):
        print(f"  [{name}] tool={msg.name}: {content}...")
    else:
        print(f"  [{name}]: {content}")

print("\nFinal answer:", result["messages"][-1].content)


## Summary

- `@tool` decorator converts a function into an agent tool; the docstring guides the agent
- `create_agent(llm, tools, system_prompt=...)` builds the agent (LangChain 1.x / LangGraph)
- Agent accepts `{"messages": [...]}` and returns `{"messages": [...]}`
- Final answer: `result["messages"][-1].content`
- Inspect tool calls by filtering `ToolMessage` instances from messages